--- Section 0: Setup ---

In [ ]:
# Colab setup: install dependencies
!pip -q install vllm sympy pandas tqdm

# Mount Google Drive to persist checkpoints and data across sessions
# Workspace = folder that contains FinReasoningAI_Colab.ipynb (fast scan under MyDrive).
# If multiple copies exist, set DRIVE_BASE manually below before re-running.
from pathlib import Path
from google.colab import drive
import os
import sys
import json
import re

drive.mount('/content/drive')

DRIVE_FALLBACK = "/content/drive/MyDrive/FinReasoningAI"


def _infer_notebook_workspace():
    """Parent folder of FinReasoningAI_Colab.ipynb on Drive (depth-limited, quick)."""
    root = Path("/content/drive/MyDrive")
    if not root.is_dir():
        return None
    candidates = []
    if (root / "FinReasoningAI_Colab.ipynb").is_file():
        candidates.append(root.resolve())
    for child in sorted(root.iterdir()):
        if not child.is_dir():
            continue
        if (child / "FinReasoningAI_Colab.ipynb").is_file():
            candidates.append(child.resolve())
        nested = child / "FinReasoningAI"
        if nested.is_dir() and (nested / "FinReasoningAI_Colab.ipynb").is_file():
            candidates.append(nested.resolve())
    uniq = []
    seen = set()
    for c in candidates:
        s = str(c)
        if s not in seen:
            seen.add(s)
            uniq.append(c)
    if len(uniq) == 1:
        return str(uniq[0])
    if len(uniq) > 1:
        print(
            "[WARN] Multiple FinReasoningAI_Colab.ipynb paths on Drive; using DRIVE_FALLBACK. "
            "Set DRIVE_BASE manually to your notebook folder."
        )
    return None


DRIVE_BASE = _infer_notebook_workspace() or DRIVE_FALLBACK
os.makedirs(DRIVE_BASE, exist_ok=True)
print(f"Drive workspace (data/outputs live here): {DRIVE_BASE}")

# Clone / update the GitHub repository (same code as the upstream project)
# Upstream: https://github.com/juankim834/FinReasoningAI
REPO_URL = "https://github.com/juankim834/FinReasoningAI.git"

WORKSPACE = DRIVE_BASE  # folder containing this notebook on Drive (or fallback)

# If the notebook already lives inside a git checkout, use that folder; else clone into WORKSPACE/FinReasoningAI
if os.path.isdir(os.path.join(WORKSPACE, ".git")):
    PROJECT_DIR = WORKSPACE
else:
    PROJECT_DIR = os.path.join(WORKSPACE, "FinReasoningAI")

if not os.path.isdir(os.path.join(PROJECT_DIR, ".git")):
    os.makedirs(WORKSPACE, exist_ok=True)
    print(f"Cloning {REPO_URL} -> {PROJECT_DIR}")
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    print(f"Repo already at {PROJECT_DIR}. Pulling latest...")
    !cd {PROJECT_DIR} && git pull

os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
print(f"Working directory: {os.getcwd()}")
!ls

# Library imports for evaluation notebook
import pandas as pd
from sympy import sympify  # noqa: F401
from tqdm.auto import tqdm
from vllm import LLM, SamplingParams

from tools.financial_tools import TOOL_REGISTRY, TOOL_SCHEMAS
from tools.number_parser import score_prediction

MODEL_PATH = "path/to/your/qlora/checkpoint"
TEST_FILE = "data/test.jsonl"
MAX_TOOL_ITERATIONS = 8
SAMPLE_SIZE = 200  # same 200 samples as before for fair comparison
OUTPUTS_DIR = Path(DRIVE_BASE) / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)


SYSTEM_TOOL_PROMPT = """You are a financial reasoning assistant.
Use tools when needed to compute values accurately.
When you decide to call a tool, output EXACTLY this format:
<tool_call>
{"tool": "calculator", "input": {"expression": "100 * 1.05"}}
</tool_call>
After receiving <tool_result>, continue reasoning and produce the final answer.
If no tool is needed, answer directly.
"""

print("Setup complete. Tools available:", list(TOOL_REGISTRY.keys()))

--- Section 1: Load Model & Test Set ---

In [ ]:
# Load model and test set
if "MODEL" not in globals() or MODEL is None:
    MODEL = LLM(model=MODEL_PATH)

sampling_params = SamplingParams(temperature=0.0, top_p=1.0, max_tokens=512)

with open(TEST_FILE, "r", encoding="utf-8") as f:
    test_samples = [json.loads(line) for line in f if line.strip()]

# Common schema assumptions: each sample has at least question/prompt, answer/gold, task
def get_task(sample):
    return sample.get("task", "unknown")

task_counts = {}
for s in test_samples:
    t = get_task(s)
    task_counts[t] = task_counts.get(t, 0) + 1

print("Task distribution:")
for task, n in sorted(task_counts.items()):
    print(f"- {task}: {n}")

assert (len(test_samples) >= SAMPLE_SIZE) or all(n >= SAMPLE_SIZE for n in task_counts.values()), (
    f"Need at least {SAMPLE_SIZE} total samples or {SAMPLE_SIZE} per task; "
    f"got total={len(test_samples)}, counts={task_counts}"
)

# Fair fixed subset: same order, first SAMPLE_SIZE
eval_samples = test_samples[:SAMPLE_SIZE]
print(f"Using {len(eval_samples)} samples for evaluation.")

--- Section 2: Reusable Inference Functions ---

In [ ]:
# 2a) run_no_tools and 2b) run_with_tools

TOOL_CALL_RE = re.compile(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", re.DOTALL)

def _ensure_model():
    global MODEL, sampling_params
    if "MODEL" not in globals() or MODEL is None:
        MODEL = LLM(model=MODEL_PATH)
    if "sampling_params" not in globals() or sampling_params is None:
        sampling_params = SamplingParams(temperature=0.0, top_p=1.0, max_tokens=512)

def _generate(prompt: str) -> str:
    _ensure_model()
    out = MODEL.generate([prompt], sampling_params=sampling_params)
    return out[0].outputs[0].text.strip()

def _build_prompt(system_prompt: str, history: list[dict]) -> str:
    parts = [f"<system>\n{system_prompt}\n</system>"]
    for msg in history:
        role = msg["role"]
        content = msg["content"]
        parts.append(f"<{role}>\n{content}\n</{role}>")
    parts.append("<assistant>\n")
    return "\n".join(parts)

def run_no_tools(prompt: str) -> str:
    """Simple single-pass vLLM inference with no tool loop."""
    full_prompt = _build_prompt("You are a helpful financial reasoning assistant.", [{"role": "user", "content": prompt}])
    return _generate(full_prompt)

def run_with_tools(prompt: str, max_iterations: int = MAX_TOOL_ITERATIONS) -> dict:
    """Run an agentic loop that executes tool calls emitted in <tool_call> tags."""
    history = [{"role": "user", "content": prompt}]
    tool_calls_made = []
    final_answer = ""

    for iteration in range(1, max_iterations + 1):
        model_prompt = _build_prompt(SYSTEM_TOOL_PROMPT, history)
        response = _generate(model_prompt)
        final_answer = response

        match = TOOL_CALL_RE.search(response)
        if not match:
            return {
                "final_answer": final_answer,
                "tool_calls_made": tool_calls_made,
                "iterations": iteration,
                "hit_limit": False,
            }

        tool_payload_raw = match.group(1)
        try:
            tool_payload = json.loads(tool_payload_raw)
            tool_name = tool_payload.get("tool")
            tool_input = tool_payload.get("input", {})

            if tool_name not in TOOL_REGISTRY:
                tool_result = {"error": f"Unknown tool: {tool_name}"}
            else:
                tool_result = TOOL_REGISTRY[tool_name](**tool_input)
        except Exception as exc:
            tool_name = None
            tool_input = {}
            tool_result = {"error": f"Tool call parse/exec failure: {exc}"}

        tool_calls_made.append({
            "tool": tool_name,
            "input": tool_input,
            "result": tool_result,
        })

        history.append({"role": "assistant", "content": response})
        history.append({
            "role": "user",
            "content": f"<tool_result>\n{json.dumps(tool_result, ensure_ascii=False)}\n</tool_result>",
        })

    return {
        "final_answer": final_answer,
        "tool_calls_made": tool_calls_made,
        "iterations": max_iterations,
        "hit_limit": True,
    }

print("Inference functions ready.")

--- Section 3: Benchmark ? No Tools ---

In [ ]:
results_dir = OUTPUTS_DIR
results_dir.mkdir(parents=True, exist_ok=True)

def _get_prompt(sample):
    return sample.get("prompt") or sample.get("question") or sample.get("input") or ""

def _get_gold(sample):
    return sample.get("gold") or sample.get("answer") or sample.get("label") or ""

no_tools_results = []
for i, sample in enumerate(tqdm(eval_samples, desc="No-tools eval")):
    prompt = _get_prompt(sample)
    gold = str(_get_gold(sample))
    pred = run_no_tools(prompt)
    score = score_prediction(pred, gold)

    row = {
        "idx": i,
        "task": sample.get("task", "unknown"),
        "prompt": prompt,
        "gold": gold,
        "pred": pred,
        **score,
    }
    no_tools_results.append(row)

with open(results_dir / "no_tools_results.json", "w", encoding="utf-8") as f:
    json.dump(no_tools_results, f, ensure_ascii=False, indent=2)

print("Task | Exact Match | Numerical Match | Parseable | N")
for task in sorted({r["task"] for r in no_tools_results}):
    rows = [r for r in no_tools_results if r["task"] == task]
    n = len(rows)
    em = sum(1 for r in rows if r["exact_match"]) / max(n, 1)
    nm = sum(1 for r in rows if r["numerical_match"]) / max(n, 1)
    parseable = sum(1 for r in rows if (r["pred_number"] is not None and r["gold_number"] is not None)) / max(n, 1)
    print(f"{task} | {em:.3f} | {nm:.3f} | {parseable:.3f} | {n}")

print(f"Saved: {results_dir / 'no_tools_results.json'}")

--- Section 4: Benchmark ? With Tools ---

In [ ]:
results_dir = OUTPUTS_DIR
results_dir.mkdir(parents=True, exist_ok=True)

with_tools_results = []
used_tool_count = 0

for i, sample in enumerate(tqdm(eval_samples, desc="With-tools eval")):
    prompt = _get_prompt(sample)
    gold = str(_get_gold(sample))
    run = run_with_tools(prompt, max_iterations=MAX_TOOL_ITERATIONS)
    pred = run["final_answer"]
    score = score_prediction(pred, gold)

    if len(run["tool_calls_made"]) > 0:
        used_tool_count += 1

    row = {
        "idx": i,
        "task": sample.get("task", "unknown"),
        "prompt": prompt,
        "gold": gold,
        "pred": pred,
        "tool_calls_made": run["tool_calls_made"],
        "iterations": run["iterations"],
        "hit_limit": run["hit_limit"],
        **score,
    }
    with_tools_results.append(row)

with open(results_dir / "with_tools_results.json", "w", encoding="utf-8") as f:
    json.dump(with_tools_results, f, ensure_ascii=False, indent=2)

print("Task | Exact Match | Numerical Match | Parseable | N")
for task in sorted({r["task"] for r in with_tools_results}):
    rows = [r for r in with_tools_results if r["task"] == task]
    n = len(rows)
    em = sum(1 for r in rows if r["exact_match"]) / max(n, 1)
    nm = sum(1 for r in rows if r["numerical_match"]) / max(n, 1)
    parseable = sum(1 for r in rows if (r["pred_number"] is not None and r["gold_number"] is not None)) / max(n, 1)
    print(f"{task} | {em:.3f} | {nm:.3f} | {parseable:.3f} | {n}")

usage_rate = used_tool_count / max(len(with_tools_results), 1)
print(f"Tool usage rate: {usage_rate * 100:.2f}% of samples triggered at least one tool call")
print(f"Saved: {results_dir / 'with_tools_results.json'}")

--- Section 5: Delta Analysis ---

In [ ]:
results_dir = OUTPUTS_DIR

with open(results_dir / "no_tools_results.json", "r", encoding="utf-8") as f:
    no_tools_results = json.load(f)
with open(results_dir / "with_tools_results.json", "r", encoding="utf-8") as f:
    with_tools_results = json.load(f)

no_by_idx = {r["idx"]: r for r in no_tools_results}
wt_by_idx = {r["idx"]: r for r in with_tools_results}

def _metrics(rows):
    n = len(rows)
    em = sum(1 for r in rows if r["exact_match"]) / max(n, 1)
    nm = sum(1 for r in rows if r["numerical_match"]) / max(n, 1)
    return em, nm

tasks = sorted(set(r["task"] for r in no_tools_results) | set(r["task"] for r in with_tools_results))
print("Task | No Tools EM | With Tools EM | Delta | No Tools NM | With Tools NM | Delta")
for task in tasks:
    nt_rows = [r for r in no_tools_results if r["task"] == task]
    wt_rows = [r for r in with_tools_results if r["task"] == task]
    nt_em, nt_nm = _metrics(nt_rows)
    wt_em, wt_nm = _metrics(wt_rows)
    print(f"{task} | {nt_em:.3f} | {wt_em:.3f} | {wt_em-nt_em:+.3f} | {nt_nm:.3f} | {wt_nm:.3f} | {wt_nm-nt_nm:+.3f}")

nt_em, nt_nm = _metrics(no_tools_results)
wt_em, wt_nm = _metrics(with_tools_results)
print(f"OVERALL | {nt_em:.3f} | {wt_em:.3f} | {wt_em-nt_em:+.3f} | {nt_nm:.3f} | {wt_nm:.3f} | {wt_nm-nt_nm:+.3f}")

used_tool = sum(1 for r in with_tools_results if len(r.get("tool_calls_made", [])) > 0)
usage_rate = used_tool / max(len(with_tools_results), 1)
print(f"Tool usage rate: {usage_rate * 100:.2f}% of samples triggered at least one tool call")

tool_freq = {}
for r in with_tools_results:
    for tc in r.get("tool_calls_made", []):
        name = tc.get("tool") or "<invalid>"
        tool_freq[name] = tool_freq.get(name, 0) + 1

print("Top 5 most frequently called tools:")
for name, count in sorted(tool_freq.items(), key=lambda x: x[1], reverse=True)[:5]:
    print(f"- {name}: {count}")

regressions = []
for idx, nt in no_by_idx.items():
    wt = wt_by_idx.get(idx)
    if wt is None:
        continue
    # Regression if with-tools loses either exact or numerical match achieved by no-tools
    if (nt["exact_match"] and not wt["exact_match"]) or (nt["numerical_match"] and not wt["numerical_match"]):
        regressions.append({
            "idx": idx,
            "task": nt["task"],
            "prompt": nt["prompt"],
            "gold": nt["gold"],
            "no_tools_pred": nt["pred"],
            "with_tools_pred": wt["pred"],
        })

print(f"Regression cases: {len(regressions)}")
for r in regressions[:20]:
    print(f"- idx={r['idx']} task={r['task']} | gold={r['gold']} | no_tools={r['no_tools_pred']} | with_tools={r['with_tools_pred']}")

--- Section 6: Failure Case Inspector ---

In [ ]:
results_dir = OUTPUTS_DIR

with open(results_dir / "no_tools_results.json", "r", encoding="utf-8") as f:
    no_tools_results = json.load(f)
with open(results_dir / "with_tools_results.json", "r", encoding="utf-8") as f:
    with_tools_results = json.load(f)

wt_by_idx = {r["idx"]: r for r in with_tools_results}

# numerical_reasoning failures in no-tools
failures = [
    r for r in no_tools_results
    if r.get("task") == "numerical_reasoning" and not r.get("numerical_match", False)
]

print(f"Inspecting up to 10 numerical_reasoning no-tools failures (found {len(failures)}).")
for r in failures[:10]:
    w = wt_by_idx.get(r["idx"], {})
    helped = bool(w) and (w.get("numerical_match", False) and not r.get("numerical_match", False))

    print("=" * 80)
    print(f"IDX: {r['idx']}")
    print(f"Question: {r['prompt']}")
    print(f"Gold: {r['gold']}")
    print(f"No-tools pred: {r['pred']}")
    print(f"No-tools extracted: pred_number={r.get('pred_number')} gold_number={r.get('gold_number')}")

    if w:
        print(f"With-tools pred: {w.get('pred')}")
        print(f"With-tools extracted: pred_number={w.get('pred_number')} gold_number={w.get('gold_number')}")
        print(f"With-tools numerical_match: {w.get('numerical_match')} | Tool helped: {helped}")
        print(f"Tool calls: {len(w.get('tool_calls_made', []))}")
    else:
        print("With-tools result missing for this index.")